# 🧠 Machine Learning Modeling

**Team Member:** Mohamed  
**Task:** Train and evaluate 6-8 machine learning models to predict startup success.

**Inputs:**
- `../data/processed/train_processed.csv`
- `../data/processed/test_processed.csv`

**Outputs:**
- `../models/model.pkl` (Best performing model)
- Feature Importance or SHAP analysis
- Evaluation Report

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score, GridSearchCV
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.ensemble import RandomForestClassifier,AdaBoostClassifier,GradientBoostingClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score, f1_score, precision_score, recall_score
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier

train_df = pd.read_csv('../data/processed/train_processed.csv')
test_df = pd.read_csv('../data/processed/test_processed.csv')

#Data Loading & Splitting
X_train = train_df.drop('status', axis=1)
y_train = train_df['status']

X_test = test_df.drop('status', axis=1)
y_test = test_df['status']

print("Training Data Shape:", X_train.shape)
print("Testing Data Shape:", X_test.shape)

## 1. Model Training & Hyperparameter Tuning

In [ ]:
models={
    "Gradient Boosting":GradientBoostingClassifier(random_state=42),
    "AdaBoost":AdaBoostClassifier(random_state=42),
    "Random Forest":RandomForestClassifier( class_weight="balanced",n_estimators=50, random_state=42),
    "Logistic Regression":LogisticRegression( class_weight="balanced", max_iter=1000 ,random_state=42),
    "Decission Tree":DecisionTreeClassifier(class_weight="balanced",random_state=42),
    "Naive Bayes":GaussianNB(),
    "KNN":KNeighborsClassifier(n_neighbors=5)
}
results=[]



In [ ]:
for name, modele in models.items():
    model = modele
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    f1=f1_score(y_test, y_pred,average="weighted")
    accuracy=accuracy_score(y_test, y_pred)
    precision=precision_score(y_test, y_pred,average="weighted",zero_division=0)
    recall=recall_score(y_test, y_pred,average="weighted")
    results.append({
        'Model': name,
        'Accuracy': accuracy,
        'Precision': precision,
        'Recall': recall,
        'F1-Score': f1
    })

results_df = pd.DataFrame(results).sort_values(by='F1-Score', ascending=False)
print(results_df.to_markdown(index=False))

In [ ]:

param_grid = {
    'n_estimators': [100, 200],
    'learning_rate': [0.05, 0.1],
    'max_depth': [3, 5]
}

gb_model = GradientBoostingClassifier(random_state=42)
grid_search = GridSearchCV(estimator=gb_model, param_grid=param_grid, cv=3, scoring='f1_weighted', n_jobs=-1, verbose=1)

grid_search.fit(X_train, y_train)

best_gb_model = grid_search.best_estimator_
print(f"Best Parameters: {grid_search.best_params_}")

best_y_pred = best_gb_model.predict(X_test)
print(classification_report(y_test, best_y_pred))

In [ ]:
# Plot Feature Importances
importances = best_gb_model.feature_importances_
indices = np.argsort(importances)[::-1][:10]
features = X_train.columns[indices]

plt.figure(figsize=(10, 6))
sns.barplot(x=importances[indices], y=features, palette='viridis')
plt.title('Top 10 Feature Importances (Gradient Boosting)')
plt.xlabel('Relative Importance')
plt.ylabel('Feature')
plt.tight_layout()

plt.savefig('feature_importance.png')
plt.show()

In [ ]:
# Save the final tuned model
joblib.dump(best_gb_model, '../models/model.pkl')
print("Model saved successfully as 'model.pkl'. Ready for Streamlit integration!")